In [1]:
import os
import sys
# import geopandas as gpd
import pandas as pd
import numpy as np
import h5py
import matplotlib.pyplot as plt
from openquake.hazardlib import site
from openquake.baselib import hdf5

Read processed np arrays from dask dataframe

In [2]:
import numpy as np
import pandas as pd
from openquake.hazardlib import site
from openquake.baselib import hdf5

true_d = np.load('true_d_dask.npy')
true_event_id = np.load('event_id_true.npy')
true_site_id = np.load('site_id_true.npy')

save as oq gmf_data in hdf5 format

In [3]:
sites_path = '/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/data/processed/lat_lon_idx_CT_892.txt'
sites = pd.read_csv(sites_path, sep=',', header=None)
sites.columns = ['m', 'n', 'lat', 'lon']
sites = sites[['lat', 'lon']]

# site collection
haz_sitecol = site.SiteCollection.from_points(sites['lon'], sites['lat'],req_site_params = {})


In [4]:
#prepare hdf5 file with gmf_data and sitecol as groups
hdf5_file = '/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/risk/demos/tsunamiebr_gmf/tsunami_hazard_gmf.hdf5'

with hdf5.File(hdf5_file, 'w') as hf:
    hf.create_group('gmf_data')
    hf.create_dataset('gmf_data/sid', data=true_site_id,dtype='uint32')
    hf.create_dataset('gmf_data/eid', data=true_event_id, dtype='uint32')
    hf.create_dataset('gmf_data/gmv_0', data=true_d, dtype='float32')
    hf['gmf_data'].attrs['__pdcolumns__'] = 'sid eid gmv_0'
    hf['gmf_data'].attrs['imts'] = 'PGA'
    hf['gmf_data'].attrs['num_events'] = 53550
    hf['gmf_data'].attrs['investigation_time'] = 1
    hf['gmf_data'].attrs['effective_time'] = 1
    hf.create_group('sitecol')
    hf['sitecol'] = haz_sitecol
    hf.close()
print('HDF5 file created successfully')

HDF5 file created successfully


In [5]:
#prepare hdf5 file with gmf_data and sitecol as groups
hdf5_file = '/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/risk/demos/tsunamiebr_imt/tsunami_hazard.hdf5'

with hdf5.File(hdf5_file, 'w') as hf:
    hf.create_group('gmf_data')
    hf.create_dataset('gmf_data/sid', data=true_site_id,dtype='uint32')
    hf.create_dataset('gmf_data/eid', data=true_event_id, dtype='uint32')
    hf.create_dataset('gmf_data/FLOWDEPTH', data=true_d, dtype='float32')
    hf['gmf_data'].attrs['__pdcolumns__'] = 'sid eid FLOWDEPTH'
    hf['gmf_data'].attrs['imts'] = 'FLOWDEPTH'
    hf['gmf_data'].attrs['num_events'] = 53550
    hf['gmf_data'].attrs['investigation_time'] = 1
    hf['gmf_data'].attrs['effective_time'] = 1
    hf.create_group('sitecol')
    hf['sitecol'] = haz_sitecol
    hf.close()
print('HDF5 file created successfully')

HDF5 file created successfully


Check hdf5 file

In [6]:
import numpy as np
import pandas as pd
from openquake.hazardlib import site
from openquake.baselib import hdf5

hdf5_file = '/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/risk/demos/tsunamiebr_imt/tsunami_hazard.hdf5'
with hdf5.File(hdf5_file, 'r') as hf:
    # gmf_data = hf['gmf_data\eid']
    # print(gmf_data)
    print(hf['gmf_data'])
    # hf.close()

<HDF5 group "/gmf_data" (3 members)>


Check hdf5 file for attributes and data groups

In [7]:
import h5py

def print_hdf5_attributes(obj, name):
    """
    A recursive function to print the attributes of an HDF5 file.
    """
    print(f"Attributes of: {name}")
    for key, val in obj.attrs.items():
        print(f"{key}: {val}")

    if isinstance(obj, h5py.Group):
        for key, item in obj.items():
            print_hdf5_attributes(item, f"{name}/{key}")

def print_hdf5_content(obj, name):
    """
    A recursive function to print the content of an HDF5 file.
    """
    if isinstance(obj, h5py.Group):
        print(f"Group: {name}")
        for key, item in obj.items():
            print_hdf5_content(item, f"{name}/{key}")
    elif isinstance(obj, h5py.Dataset):
        print(f"Dataset: {name}")
        print(f"Shape: {obj.shape}, Dtype: {obj.dtype}")
        print("Data:")
        print(obj[:])  # Print the actual data
    else:
        print(f"Unknown type: {name}")

with h5py.File(hdf5_file, "r") as f:
    print_hdf5_content(f, '/')
    print_hdf5_attributes(f, '/')


Group: /
Group: //gmf_data
Dataset: //gmf_data/FLOWDEPTH
Shape: (2421054,), Dtype: float32
Data:
[ 18.  17.  14. ... 108.  90.  53.]
Dataset: //gmf_data/eid
Shape: (2421054,), Dtype: uint32
Data:
[0 0 0 ... 9 9 9]
Dataset: //gmf_data/sid
Shape: (2421054,), Dtype: uint32
Data:
[     2      3      4 ... 579390 579407 579423]
Group: //sitecol
Dataset: //sitecol/depth
Shape: (579439,), Dtype: float64
Data:
[0. 0. 0. ... 0. 0. 0.]
Dataset: //sitecol/lat
Shape: (579439,), Dtype: float64
Data:
[37.31034 37.31034 37.31034 ... 37.51062 37.51062 37.51062]
Dataset: //sitecol/lon
Shape: (579439,), Dtype: float64
Data:
[15.07406 15.07417 15.07429 ... 15.10487 15.10499 15.1051 ]
Dataset: //sitecol/sids
Shape: (579439,), Dtype: uint32
Data:
[     0      1      2 ... 579436 579437 579438]
Attributes of: /
Attributes of: //gmf_data
__pdcolumns__: sid eid FLOWDEPTH
effective_time: 1
imts: FLOWDEPTH
investigation_time: 1
num_events: 53550
Attributes of: //gmf_data/FLOWDEPTH
Attributes of: //gmf_data/eid


In [8]:
with h5py.File(hdf5_file, "r") as f:
    print(f)
    #get the keys of the hdf5 file
    for key in f.keys():
        print(key)
        #get the keys of the group
        for key1 in f[key].keys():
            print('\t',key1)
            

<HDF5 file "tsunami_hazard.hdf5" (mode r)>
gmf_data
	 FLOWDEPTH
	 eid
	 sid
sitecol
	 depth
	 lat
	 lon
	 sids
